# Task 20: Spatiotemporal Forecasting Training Orchestrator
This notebook trains the CYCLO-NEXUS Phase 3 ConvLSTM + Bi-GRU model.

In [ ]:
!pip install torch torchvision numpy tqdm

In [ ]:
from google.colab import drive
import sys
import torch

# Mount Drive and set path
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/cyclonegaurd')

In [ ]:
from pathlib import Path
from forecaster.data.temporal_matrix_builder import TemporalMatrixBuilder
from torch.utils.data import Dataset, DataLoader

# 1. Define custom PyTorch Dataset leveraging our strict Chronological Builder
class CycloneSequenceDataset(Dataset):
    def __init__(self, data_dir):
        # Load and build valid sequences of length 6 (3 hours of 30-min data)
        file_paths = list(Path(data_dir).glob('*.npy'))
        self.sequences = TemporalMatrixBuilder.build_sequences(file_paths, seq_length=6)
        
        # Mocking targets for orchestration blueprint
        self.targets = [{'track_delta': torch.tensor([0.1, -0.05]), 'v_max': torch.tensor([85.0]), 'dp': torch.tensor([12.0])} for _ in self.sequences]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        # Sequence shape: (6, 4, 1024, 1024)
        return self.sequences[idx], self.targets[idx]

print("Building dataset and validating chronological leakage invariants...")
dataset = CycloneSequenceDataset('/content/drive/MyDrive/cyclonegaurd/data/tensors/')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)
print(f"Loaded {len(dataset)} valid sequences.")

In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from forecaster.models.spatiotemporal_forecaster import CycloneForecaster

# 2. Initialize Model and Training Components
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CycloneForecaster(in_channels=4, conv_hidden=64, gru_hidden=128).to(device)

# Use AdamW with weight decay to prevent overfitting on complex spatiotemporal features
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# HuberLoss is robust to outliers in trajectory offsets and intensity spikes
track_criterion = nn.HuberLoss()
vmax_criterion = nn.MSELoss()
dp_criterion = nn.MSELoss()

# 3. Execute the Heavy PyTorch Training Loop
epochs = 30
print(f"Starting training on {device} for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    for sequences, targets in tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}'):
        sequences = sequences.to(device)  # (Batch, 6, 4, 1024, 1024)
        track_gt = targets['track_delta'].to(device)
        vmax_gt = targets['v_max'].to(device)
        dp_gt = targets['dp'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (executes CNN Stem -> ConvLSTM -> Bi-GRU)
        predictions = model(sequences)
        
        # Multi-task loss computation
        loss_track = track_criterion(predictions['track_delta'], track_gt)
        loss_vmax = vmax_criterion(predictions['v_max_pred'], vmax_gt)
        loss_dp = dp_criterion(predictions['dp_pred'], dp_gt)
        
        # Combined objective (weighted depending on task importance)
        loss = loss_track + 0.5 * loss_vmax + 0.5 * loss_dp
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{epochs}] - Avg Loss: {avg_loss:.4f}")


In [ ]:
# 4. Export trained weights
save_path = '/content/drive/MyDrive/cyclonegaurd/weights/forecaster_weights.pth'
torch.save(model.state_dict(), save_path)
print(f"Training complete. Model weights saved to {save_path}")